# Week 02｜いま起きている地震を地図にする

最後の30分に始め、家で自分の地図として仕上げるNotebookです。まずYohのデモを見て、コードを実行する**前**に結果を予想してください。地震データは[USGSの過去24時間CSV](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)を読みます。ネット接続がない場合は、同じフォルダに置いた`all-month-earthquakes.csv`（2024年の予備データ）を使います。

今日の目標：データの列が地図の何になるかを説明し、少なくとも一つの設定を自分で変える。ループの詳しい書き方は後の週で学びます。

## 1｜予想してから実行

| CSVの列 | 地図では何になると思う？ |
| --- | --- |
| `latitude`、`longitude` | ここに書く |
| `mag` | ここに書く |
| `place` | ここに書く |

大きな地震だけ表示したら、地図はどう変わるでしょう？　一文で予想してください。

## 2｜地図に必要な道具

最初の一度だけ下のCellを実行します。`pandas`は表を読み、`folium`は地図を作ります。インストールで止まったら、まずKernelが`.venv`かを確認してください。

In [ ]:
%pip install pandas folium

## 3｜データを読む

ライブデータを試し、失敗したときだけ2024年のCSVに切り替えます。`source`に、実際に使ったデータの名前を保存します。地震の大きさや緯度・経度が空欄の行は除きます。

In [ ]:
from pathlib import Path
from html import escape
import pandas as pd
import folium

feed_url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.csv"
backup_file = Path("all-month-earthquakes.csv")

try:
    raw = pd.read_csv(feed_url)
    source = "USGS・過去24時間（ライブ）"
except Exception as error:
    if not backup_file.exists():
        raise FileNotFoundError("ネットに接続するか、予備CSVをNotebookと同じフォルダに置いてください") from error
    raw = pd.read_csv(backup_file)
    source = "2024年10月の予備データ"

for column in ["latitude", "longitude", "mag"]:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")
data = raw.dropna(subset=["latitude", "longitude", "mag"]).copy()
print(f"{source}：{len(data)}件読み込みました")
data[["time", "place", "mag", "latitude", "longitude"]].head()

## 4｜自分で選ぶ設定

**ここを変えると地図が変わります。** `minimum_magnitude`は表示する最小の大きさ、`marker_color`は点の色、`center`は地図の中心です。`max_events`は重くならないよう表示数を制限します。まずはそのまま実行し、次に一つ変えてください。

In [ ]:
map_title = "過去24時間の地震"  # 自分のタイトルに変える
minimum_magnitude = 2.5      # 例：4.0なら大きめの地震だけ
marker_color = "#285f87"    # 例："#c15d3a"
center = [20, 0]             # [緯度, 経度]
max_events = 200            # 表示する最大件数

events = data[data["mag"] >= minimum_magnitude].head(max_events)
print(f"{map_title}：{len(events)}件を表示します")

## 5｜地図を作る

`folium.Map`で地図を用意し、各行の緯度・経度に丸を置きます。丸の大きさは`mag`で変わります。下のCode Cellは最初はそのまま使って構いません。`for`の仕組みはWeek 05で学びます。

In [ ]:
earthquake_map = folium.Map(location=center, zoom_start=2)

for _, row in events.iterrows():
    place = escape(str(row["place"]))
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4 + row["mag"] * 2,
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.6,
        popup=f"{place}：M{row['mag']:.1f}"
    ).add_to(earthquake_map)

earthquake_map

## 6｜結果を読む

- 予想と同じだったこと：
- 意外だったこと：
- 変更した設定と、地図がどう変わったか：
- 今日使えたのはライブ／予備のどちらのデータ？：

**持ち帰り課題：** 誰に何を見せる地図かを決め、設定を二つ以上変えてタイトルを付けます。Notebookを上から実行し直し、変更理由と地図から読み取れたことをMarkdownで書きます。10月14日（水）23:59 JSTまでに`.ipynb`をUNIPAへ提出。予備CSVを使った場合は、そのCSVも一緒に提出してください。